<a href="https://colab.research.google.com/github/Slmpire/DSN_AI_Bootcamp_2026/blob/main/Agentic_Loan_Inquiry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Create a RAG-enabled WhatsApp Loan Inquiry Agent in Google Colab.

To achieve this, perform the following steps:
1. **Secure API Key Setup**: Demonstrate how to retrieve the OpenAI API Key securely using `google.colab.userdata` to avoid hardcoding secrets.
2. **Install Dependencies**: Install necessary packages including `openai`, `flask`, `pyngrok`, and `chromadb`.
3. **Infuse RAG Concept**: Build a step-by-step beginner guide and Python code using a simple vector store (like ChromaDB) to store, search, and retrieve policy documents for the loan agent.
4. **Build the Loan Inquiry Agent**: Program the core AI agent using the OpenAI API, configuring it with system prompts that utilize the retrieved RAG contexts.
5. **Test Agent Responses Locally**: Implement an interactive local CLI loop to verify the agent's accuracy and behavior using the retrieved context.
6. **Set Up Webhook for WhatsApp**: Write a Flask application that receives Twilio WhatsApp webhook payloads, processes them through the RAG loan agent, and formats the response.
7. **Expose Webhook and Deploy Guide**: Use `pyngrok` to expose the local Flask server to a public URL, and write a clear, step-by-step guide on how to configure this URL in the Twilio Sandbox for WhatsApp.

## Secure API Key Setup

### Subtask:
Demonstrate how to securely retrieve the OpenAI API Key in Google Colab to avoid hardcoding credentials.


**Reasoning**:
Create a python code cell to safely retrieve the OpenAI API Key from Colab Secrets using userdata, wrapping it in a try-except block to handle missing keys gracefully.



In [ ]:
try:
    from google.colab import userdata
    openai_api_key = userdata.get('OPENAI_API_KEY')
    print("Successfully retrieved OpenAI API Key from Colab Secrets.")
except Exception as e:
    openai_api_key = None
    print("Could not retrieve 'OPENAI_API_KEY' from Colab Secrets.")
    print("Please add your API key by clicking the key icon (Secrets tab) on the left sidebar, adding a new secret named 'OPENAI_API_KEY', and turning on notebook access.")

Successfully retrieved OpenAI API Key from Colab Secrets.


## Install Dependencies

### Subtask:
Install required Python libraries including openai, flask, pyngrok, and chromadb to support our RAG WhatsApp Agent.


**Reasoning**:
Install the required Python libraries for the project, which are openai, flask, pyngrok, and chromadb, using pip.



In [ ]:
!pip install -q openai flask pyngrok chromadb pysqlite3-binary
# Force chromadb to use the newer pysqlite3 binary
import sys
try:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass
print("Dependencies installed successfully and sqlite3 patched.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

## Infuse RAG (Retrieval-Augmented Generation) Concept

### Subtask:
Build a step-by-step beginner guide and Python code using a simple vector store (like ChromaDB) to store, search, and retrieve policy documents for the loan agent.


### Introduction to Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** is a technique used to improve the accuracy and reliability of generative AI models by fetching relevant facts from an external knowledge base before generating a response.

Instead of relying solely on the pre-trained knowledge inside the Large Language Model (LLM)—which might be outdated or lack private company information—we perform a two-step process:
1. **Retrieval**: Given a user query (e.g., *"What is the maximum loan amount?"*), we search a database (a Vector Store) containing our specific loan policy documents to find the most relevant snippets.
2. **Generation**: We feed both the user's query and the retrieved documents (context) into the LLM, instructing it to answer the query *only* based on the retrieved facts.

### Vector Stores and Embeddings
To search text semantically, we convert our text documents into **embeddings** (numerical representations of meaning). We store these embeddings in **ChromaDB**, an open-source vector database. When a user asks a question, ChromaDB compares the embedding of the question against the stored documents to retrieve the closest semantic matches.

**Reasoning**:
Create and execute the Python code to initialize ChromaDB, add sample loan policy documents, and define a retrieval helper function.



In [ ]:
import sys
# Force sqlite3 to point to pysqlite3-binary for chromadb compatibility in Google Colab
try:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

import chromadb

# 1. Initialize an in-memory ChromaDB client
chroma_client = chromadb.Client()

# 2. Create a new collection named 'loan_policies'
collection = chroma_client.get_or_create_collection(name="loan_policies")

# 3. Define sample loan policy documents
policies = [
    {
        "id": "doc1",
        "text": "The Personal Loan policy offers loan amounts from $5,000 to a maximum of $50,000. The interest rates are fixed and range from 5.99% to 15.99% APR, depending on the applicant's credit score.",
        "metadata": {"category": "personal_loan", "topic": "limitations"}
    },
    {
        "id": "doc2",
        "text": "To be eligible for a personal or business loan, applicants must be at least 18 years old, provide proof of a steady monthly income of at least $2,500, and have a minimum credit score of 620.",
        "metadata": {"category": "general_eligibility", "topic": "criteria"}
    },
    {
        "id": "doc3",
        "text": "Our Home Renovation Loan program offers financing up to $100,000. The standard repayment period options are 5, 10, or 15 years, with no pre-payment penalties.",
        "metadata": {"category": "home_loan", "topic": "repayment"}
    },
    {
        "id": "doc4",
        "text": "Required documentation for loan processing includes: a valid government-issued ID, the last 3 months of bank statements, and the most recent W-2 form or tax return.",
        "metadata": {"category": "general_eligibility", "topic": "documentation"}
    }
]

# 4. Add documents to the ChromaDB collection using the default embedding function
collection.add(
    documents=[doc["text"] for doc in policies],
    metadatas=[doc["metadata"] for doc in policies],
    ids=[doc["id"] for doc in policies]
)

# 5. Helper function to retrieve the most relevant policy text matching a query
def retrieve_context(query, n_results=2):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    # Extract and join the retrieved documents into a single cohesive string
    retrieved_docs = results["documents"][0]
    return "\n\n".join(retrieved_docs)

# Verify the retrieval system works with a test query
test_query = "What documentation do I need to apply for a loan and what is the credit score criteria?"
retrieved_context = retrieve_context(test_query, n_results=2)
print("--- Test Retrieval --- ")
print(f"Query: {test_query}")
print("Retrieved Context:")
print(retrieved_context)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 13.8MiB/s]


--- Test Retrieval --- 
Query: What documentation do I need to apply for a loan and what is the credit score criteria?
Retrieved Context:
Required documentation for loan processing includes: a valid government-issued ID, the last 3 months of bank statements, and the most recent W-2 form or tax return.

To be eligible for a personal or business loan, applicants must be at least 18 years old, provide proof of a steady monthly income of at least $2,500, and have a minimum credit score of 620.


**Reasoning**:
The previous cell completed successfully despite downloading output in stderr. I will now proceed to the next step, which is to build and program the core AI loan inquiry agent using the OpenAI API configured with a custom system prompt.



In [ ]:
from openai import OpenAI

# Initialize OpenAI Client
client = OpenAI(api_key=openai_api_key)

def ask_loan_agent(user_query):
    # 1. Retrieve the relevant context from our vector database
    context = retrieve_context(user_query, n_results=2)

    # 2. Define the system and user messages
    system_prompt = (
        "You are a helpful and professional WhatsApp Loan Inquiry Agent.\n"
        "Answer the user's questions based ONLY on the provided context. If the "
        "answer cannot be found in the context, politely inform the user that you "
        "do not have that information and suggest contacting customer support.\n\n"
        f"### Context:\n{context}"
    )

    try:
        # 3. Call the OpenAI Chat Completions API
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {str(e)}"

# Test the agent with a sample query
sample_query = "Can I get a loan if my credit score is 600?"
print(f"User: {sample_query}")
print(f"Agent: {ask_loan_agent(sample_query)}")

User: Can I get a loan if my credit score is 600?
Agent: I'm sorry, but based on the information provided, the minimum credit score required for a loan is 620. Therefore, with a credit score of 600, you would not meet the eligibility criteria for a loan.


## Build the Loan Inquiry Agent

### Subtask:
Create the core Agentic AI logic using the OpenAI API, configuring it with system prompts that utilize the retrieved RAG contexts.


## Test Agent Responses Locally

### Subtask:
Implement an interactive local CLI loop to verify the agent's accuracy and multi-turn behavior using the retrieved context.

In [ ]:
def local_interactive_loop():
    print("============================================================")
    print("💬 Welcome to the RAG Loan Inquiry Agent Interactive CLI!")
    print("Type 'exit' or 'quit' to terminate the session.")
    print("============================================================\n")

    while True:
        user_query = input("👤 You: ")
        if user_query.strip().lower() in ['exit', 'quit']:
            print("🤖 Agent: Thank you for your inquiry! Have a great day!")
            break

        if not user_query.strip():
            continue

        response = ask_loan_agent(user_query)
        print(f"🤖 Agent: {response}\n")

# Run the CLI Loop (Type some test queries like 'What is the maximum loan amount?')
# Note: Run this cell and check the prompt box below to interact.
local_interactive_loop()

💬 Welcome to the RAG Loan Inquiry Agent Interactive CLI!
Type 'exit' or 'quit' to terminate the session.

👤 You: quit
🤖 Agent: Thank you for your inquiry! Have a great day!


## Set Up Webhook for WhatsApp

### Subtask:
Write a Flask application that receives Twilio WhatsApp webhook payloads (sent via standard `application/x-www-form-urlencoded` post requests), extracts the message body, processes it with our RAG agent, and replies back with a Twilio-compatible XML (TwiML) format payload.

In [ ]:
from flask import Flask, request
import urllib.parse

app = Flask(__name__)

@app.route("/webhook", methods=['POST'])
def whatsapp_webhook():
    # Twilio sends incoming WhatsApp data as form-encoded data
    incoming_msg = request.values.get('Body', '').strip()
    sender_number = request.values.get('From', '')

    print(f"📩 Received message from {sender_number}: '{incoming_msg}'")

    # Generate a response using our RAG core agent
    agent_reply = ask_loan_agent(incoming_msg)

    # Construct Twilio's TwiML XML payload to reply back
    twiml_response = f"""<?xml version="1.0" encoding="UTF-8"?>
<Response>
    <Message>{agent_reply}</Message>
</Response>"""

    return twiml_response, 200, {'Content-Type': 'text/xml'}

print("Flask app initialized with webhooks routing to '/webhook'")

Flask app initialized with webhooks routing to '/webhook'


## Expose Webhook and Deploy Guide

### Subtask:
Use `pyngrok` to securely expose the local Flask server running inside our Colab container to a public dynamic URL, and configure it inside the Twilio Sandbox.

In [ ]:
import os
from pyngrok import ngrok
import threading

# Securely fetch NGROK_AUTHTOKEN from Google Secrets if configured
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print("Successfully configured ngrok auth token.")
    else:
        print("No NGROK_AUTHTOKEN found. Working with free limited requests if possible or configure secrets.")
except Exception:
    print("Could not configure optional ngrok auth token.")

def run_flask():
    # Running Flask on a separate background thread to avoid locking Colab cells
    app.run(port=5000, host="0.0.0.0", use_reloader=False)

# Start background Flask server thread
flask_thread = threading.Thread(target=run_flask)
flask_thread.start()

# Connect public ngrok tunnel on port 5000
try:
    public_url = ngrok.connect(5000)
    print("\n=========================================================")
    print("🚀 Agentic Webhook Tunnel is LIVE!")
    print(f"🔗 Public Webhook URL: {public_url.public_url}/webhook")
    print("=========================================================")
except Exception as e:
    print(f"Failed to start ngrok tunnel: {e}")

Successfully configured ngrok auth token.
 * Serving Flask app '__main__'
 * Debug mode: off

🚀 Agentic Webhook Tunnel is LIVE!
🔗 Public Webhook URL: https://8fcc-34-21-165-229.ngrok-free.app/webhook


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


### 🛠️ Step-by-Step Twilio Sandbox Configuration Instructions

Now that your secure pipeline is running, use these steps to map it to WhatsApp:

1. **Get a Twilio Account**: Sign up at [Twilio](https://www.twilio.com/) if you haven't already.
2. **Access Sandbox**: Go to the **Develop > Messaging > Try it Out > Send a WhatsApp Message** menu within your Twilio Console.
3. **Opt-In**: Join the Sandbox by scanning the QR code or sending the specified text (e.g., `join <your-sandbox-keyword>`) to the designated Twilio WhatsApp number from your smartphone.
4. **Configure Webhook**:
   - Under **Sandbox settings** on the same Twilio page, locate the **"WHEN A MESSAGE COMES IN"** input field.
   - Copy the public `https://...` URL printed by the code cell above (ending in `/webhook`).
   - Paste it into the input field and keep the method set to **HTTP POST**.
   - Click **Save**.
5. **Simulate**: Text your newly linked Twilio WhatsApp number questions such as:
   - *"What is the maximum loan amount and interest rates?"*
   - *"Can I get a loan if my credit score is 600?"*

Your Colab workspace will capture the request, retrieve the database context, process it with OpenAI, and reply in real-time straight to your phone!

In [ ]:
def stop_all_tunnels():
    """
    Utility function to safely shut down the active ngrok tunnels
    and free up the port when you are finished testing.
    """
    from pyngrok import ngrok
    try:
        tunnels = ngrok.get_tunnels()
        for tunnel in tunnels:
            print(f"Stopping tunnel: {tunnel.public_url}")
            ngrok.disconnect(tunnel.public_url)
        ngrok.kill()
        print("Successfully closed all active ngrok tunnels.")
    except Exception as e:
        print(f"An error occurred while stopping tunnels: {e}")

# Uncomment the line below to shut down tunnels when you are done:
# stop_all_tunnels()

In [ ]:
# Final Verification: Let's run a batch of test queries against our RAG model
test_questions = [
    "What is the maximum amount I can borrow for a personal loan?",
    "What documents do I need to prepare?",
    "Do you offer any options for home renovation?"
]

print("--- Batch Testing Loan Inquiry Agent ---\n")
for question in test_questions:
    print(f"Question: {question}")
    response = ask_loan_agent(question)
    print(f"Agent Response: {response}")
    print("-" * 40)


--- Batch Testing Loan Inquiry Agent ---

Question: What is the maximum amount I can borrow for a personal loan?
Agent Response: The maximum amount you can borrow for a personal loan is $50,000.
----------------------------------------
Question: What documents do I need to prepare?
Agent Response: To apply for a loan, you will need to prepare a valid government-issued ID, the last 3 months of bank statements, and either the most recent W-2 form or tax return. Additionally, you must meet the eligibility criteria of being at least 18 years old, having a steady monthly income of at least $2,500, and a minimum credit score of 620.
----------------------------------------
Question: Do you offer any options for home renovation?
Agent Response: Yes, we offer a Home Renovation Loan program that provides financing up to $100,000.
----------------------------------------
